# Pertemuan 10 — Penalaran Berbasis Aturan dan Sistem Pakar Dasar
**PIF403 Pengantar Kecerdasan Buatan · Prodi Informatika Universitas Adzkia**

Minggu lalu kita menyimpan pengetahuan. Sekarang kita **menalar** dengannya secara efisien. Sistem berbasis aturan memisahkan dua hal: **basis pengetahuan** (aturan JIKA–MAKA yang ditulis pakar) dan **mesin inferensi** (program umum yang menjalankan aturan itu).

**Setelah menyelesaikan notebook ini Anda mampu:**
1. menyusun aturan produksi JIKA–MAKA;
2. mengimplementasikan forward chaining dan backward chaining;
3. menerapkan strategi *conflict resolution*;
4. menghasilkan **jejak penjelasan** (*explanation trace*) atas setiap kesimpulan.

Notebook ini adalah **log implementasi** menuju asesmen A6 (mini-project).

In [ ]:
NAMA = "tulis nama Anda"
NIM = "tulis NIM Anda"
print(f"Notebook P10 dikerjakan oleh: {NAMA} ({NIM})")

import pandas as pd

## 1. Aturan produksi

Setiap aturan berbentuk **JIKA** *semua premis benar* **MAKA** *kesimpulan benar*. Kita lanjutkan domain diagnosis gangguan laptop dari pertemuan 9.

Setiap aturan diberi `id`, `prioritas`, dan `alasan`. Kolom `alasan` kelak dipakai untuk menjelaskan kesimpulan kepada pengguna.

In [ ]:
ATURAN = [
    dict(id="R1", jika=["tidak_menyala", "lampu_daya_mati"], maka="masalah_daya", prioritas=5,
         alasan="Laptop tidak menyala dan lampu daya mati menunjukkan arus listrik tidak sampai ke papan induk."),
    dict(id="R2", jika=["masalah_daya", "lampu_charger_mati"], maka="adaptor_rusak", prioritas=5,
         alasan="Gangguan daya disertai lampu charger mati mengarah ke adaptor."),
    dict(id="R3", jika=["masalah_daya", "lampu_charger_menyala"], maka="baterai_rusak", prioritas=5,
         alasan="Adaptor bekerja tetapi daya tidak tersimpan, sehingga baterai dicurigai."),
    dict(id="R4", jika=["layar_gelap", "kipas_berputar"], maka="masalah_tampilan", prioritas=5,
         alasan="Mesin hidup (kipas berputar) tetapi layar gelap berarti gangguan pada jalur tampilan."),
    dict(id="R5", jika=["masalah_tampilan", "monitor_eksternal_normal"], maka="panel_layar_rusak", prioritas=5,
         alasan="Gambar muncul di monitor eksternal, jadi kartu grafis sehat dan panel laptop yang bermasalah."),
    dict(id="R6", jika=["masalah_tampilan", "bunyi_beep"], maka="ram_bermasalah", prioritas=8,
         alasan="Bunyi beep saat menyala adalah kode galat memori."),
    dict(id="R7", jika=["mati_mendadak", "bodi_panas"], maka="overheat", prioritas=5,
         alasan="Mati mendadak saat bodi panas adalah mekanisme pengaman suhu."),
    dict(id="R8", jika=["overheat", "kipas_berisik"], maka="kipas_kotor", prioritas=5,
         alasan="Kipas berisik saat panas menandakan debu menumpuk pada pendingin."),
    dict(id="R9", jika=["adaptor_rusak"], maka="saran_ganti_adaptor", prioritas=3,
         alasan="Adaptor rusak tidak dapat diperbaiki dengan aman."),
    dict(id="R10", jika=["baterai_rusak"], maka="saran_ganti_baterai", prioritas=3,
         alasan="Baterai yang tidak menyimpan daya perlu diganti."),
    dict(id="R11", jika=["kipas_kotor"], maka="saran_bersihkan_pendingin", prioritas=3,
         alasan="Membersihkan kipas dan mengganti pasta termal memulihkan pendinginan."),
    dict(id="R12", jika=["overheat"], maka="saran_gunakan_alas_pendingin", prioritas=1,
         alasan="Alas pendingin membantu sirkulasi udara sebagai langkah sementara."),
    dict(id="R13", jika=["ram_bermasalah"], maka="saran_pasang_ulang_ram", prioritas=3,
         alasan="Kontak RAM yang longgar atau kotor sering pulih setelah dipasang ulang."),
    dict(id="R14", jika=["panel_layar_rusak"], maka="saran_ganti_panel", prioritas=3,
         alasan="Panel layar yang rusak harus diganti di pusat servis."),
]
tabel_aturan = pd.DataFrame([{"id": a["id"], "JIKA": " DAN ".join(a["jika"]), "MAKA": a["maka"], "prioritas": a["prioritas"]} for a in ATURAN])
print(tabel_aturan.to_string(index=False))

## 2. Forward chaining (penalaran maju, *data-driven*)

Berangkat dari **fakta** yang diketahui, jalankan aturan yang premisnya terpenuhi, tambahkan kesimpulannya sebagai fakta baru, lalu ulangi sampai tidak ada lagi aturan yang bisa dijalankan.

Satu putaran terdiri atas tiga langkah, disebut siklus **match – resolve – act**:
1. **Match**: kumpulkan semua aturan yang premisnya terpenuhi dan belum dijalankan (*conflict set*).
2. **Resolve**: pilih satu aturan dari conflict set.
3. **Act**: jalankan aturan itu (tambahkan kesimpulannya ke memori kerja).

In [ ]:
def pilih_urutan(konflik, memori):
    """Strategi paling sederhana: aturan yang paling atas di daftar."""
    return konflik[0]

def forward_chaining(fakta_awal, aturan=ATURAN, pilih=pilih_urutan, jejak=True):
    memori = list(fakta_awal)                 # memori kerja (urut sesuai waktu masuk)
    asal = {f: None for f in fakta_awal}       # fakta -> aturan yang menghasilkannya (None = pengamatan)
    sudah = set(); log = []
    putaran = 0
    while True:
        konflik = [a for a in aturan if a["id"] not in sudah
                   and all(p in memori for p in a["jika"]) and a["maka"] not in memori]
        if not konflik:
            break
        putaran += 1
        terpilih = pilih(konflik, memori)
        sudah.add(terpilih["id"]); memori.append(terpilih["maka"]); asal[terpilih["maka"]] = terpilih
        log.append(terpilih["id"])
        if jejak:
            print(f"Putaran {putaran}: conflict set = {[a['id'] for a in konflik]} → jalankan {terpilih['id']} "
                  f"→ fakta baru: {terpilih['maka']}")
    return memori, asal, log

gejala = ["tidak_menyala", "lampu_daya_mati", "lampu_charger_mati"]
memori, asal, log = forward_chaining(gejala)
print("\nMemori kerja akhir:", memori)

## 3. Jejak penjelasan (explanation trace)

Kesimpulan tanpa alasan tidak layak dipercaya. Karena kita mencatat aturan asal setiap fakta, sistem dapat menjawab pertanyaan **"mengapa Anda menyimpulkan itu?"** dengan menelusuri mundur rantai aturannya.

In [ ]:
def jelaskan(fakta, asal, tingkat=0):
    """Menjelaskan sebuah kesimpulan secara rekursif sampai ke fakta pengamatan."""
    inden = "    " * tingkat
    aturan = asal.get(fakta)
    if aturan is None:
        print(f"{inden}• {fakta}  [pengamatan pengguna]")
        return
    print(f"{inden}• {fakta}  [aturan {aturan['id']}: {aturan['alasan']}]")
    for premis in aturan["jika"]:
        jelaskan(premis, asal, tingkat + 1)

print("Mengapa sistem menyarankan mengganti adaptor?\n")
jelaskan("saran_ganti_adaptor", asal)

Jejak ini adalah bentuk paling dasar dari **explainable AI**. Sistem berbasis aturan unggul di sini: setiap kesimpulan dapat ditelusuri ke aturan yang ditulis manusia dan ke fakta yang diamati.

## 4. Conflict resolution

Bila beberapa aturan sama-sama memenuhi syarat, mana yang dijalankan lebih dulu? Strategi yang umum:

| Strategi | Aturan yang dipilih |
|---|---|
| Urutan | yang paling atas di daftar aturan |
| Prioritas (*salience*) | yang prioritasnya paling tinggi |
| Spesifisitas | yang premisnya paling banyak (paling khusus) |
| Kebaruan (*recency*) | yang memakai fakta paling baru di memori kerja |

Perbedaannya terlihat ketika beberapa gangguan muncul bersamaan.

In [ ]:
def pilih_prioritas(konflik, memori):
    return max(konflik, key=lambda a: a["prioritas"])

def pilih_spesifik(konflik, memori):
    return max(konflik, key=lambda a: len(a["jika"]))

def pilih_baru(konflik, memori):
    return max(konflik, key=lambda a: max(memori.index(p) for p in a["jika"]))

gejala_rumit = ["mati_mendadak", "bodi_panas", "kipas_berisik", "layar_gelap", "kipas_berputar", "bunyi_beep"]
baris = []
for nama, strategi in [("urutan", pilih_urutan), ("prioritas", pilih_prioritas),
                       ("spesifisitas", pilih_spesifik), ("kebaruan", pilih_baru)]:
    memori2, asal2, log2 = forward_chaining(gejala_rumit, pilih=strategi, jejak=False)
    baris.append({"strategi": nama, "urutan aturan dijalankan": " → ".join(log2)})
print(pd.DataFrame(baris).to_string(index=False))

Pada sistem **monoton** seperti ini (fakta hanya bertambah, tidak pernah dicabut), semua strategi berakhir pada himpunan kesimpulan yang sama. Yang berbeda adalah **urutannya**. Urutan menjadi penting ketika:
- sistem harus menampilkan **saran paling penting lebih dulu**;
- aturan dapat **menghapus atau mengubah** fakta;
- sistem dihentikan begitu diagnosis pertama ditemukan.

## 5. Backward chaining (penalaran mundur, *goal-driven*)

Berangkat dari **hipotesis** yang ingin dibuktikan. Cari aturan yang kesimpulannya adalah hipotesis itu, lalu jadikan premis-premisnya sebagai sub-tujuan baru. Ulangi sampai seluruh sub-tujuan bertemu fakta.

Backward chaining cocok untuk **konsultasi**: sistem hanya menanyakan gejala yang relevan dengan hipotesis yang sedang diperiksa.

In [ ]:
def backward_chaining(tujuan, fakta, aturan=ATURAN, tingkat=0, jejak=True, jalur=()):
    inden = "    " * tingkat
    if tujuan in fakta:
        if jejak: print(f"{inden}✓ {tujuan} (fakta)")
        return True
    if tujuan in jalur:                                   # mencegah putaran tak berujung
        return False
    kandidat = [a for a in aturan if a["maka"] == tujuan]
    if not kandidat:
        if jejak: print(f"{inden}✗ {tujuan} (bukan fakta, tidak ada aturan yang menyimpulkannya)")
        return False
    for a in kandidat:
        if jejak: print(f"{inden}? {tujuan} ← coba {a['id']}: perlu {a['jika']}")
        if all(backward_chaining(p, fakta, aturan, tingkat + 1, jejak, jalur + (tujuan,)) for p in a["jika"]):
            if jejak: print(f"{inden}✓ {tujuan} terbukti lewat {a['id']}")
            return True
    if jejak: print(f"{inden}✗ {tujuan} tidak terbukti")
    return False

print("Hipotesis: adaptor_rusak\n")
terbukti = backward_chaining("adaptor_rusak", gejala)
print("\nHipotesis: baterai_rusak\n")
terbukti = backward_chaining("baterai_rusak", gejala)

### Forward atau backward?

| | Forward chaining | Backward chaining |
|---|---|---|
| Titik berangkat | fakta | hipotesis |
| Pertanyaan yang dijawab | "Apa saja yang bisa disimpulkan?" | "Apakah X benar?" |
| Cocok untuk | pemantauan, peringatan dini, konfigurasi | diagnosis, konsultasi, tanya jawab |
| Risiko | menurunkan banyak fakta yang tidak diperlukan | mengulang pembuktian sub-tujuan yang sama |

## 6. Prototipe sistem pakar
Kita gabungkan semuanya menjadi fungsi `konsultasi`: menerima gejala, menjalankan inferensi, lalu mengembalikan diagnosis, saran, **dan** penjelasannya.

In [ ]:
DIAGNOSIS = {"adaptor_rusak", "baterai_rusak", "panel_layar_rusak", "ram_bermasalah", "overheat", "kipas_kotor"}

def konsultasi(gejala, strategi=pilih_prioritas):
    memori, asal, log = forward_chaining(gejala, pilih=strategi, jejak=False)
    diagnosis = [f for f in memori if f in DIAGNOSIS]
    saran = [f for f in memori if f.startswith("saran_")]
    print("Gejala    :", ", ".join(gejala))
    if not diagnosis:
        print("Diagnosis : belum dapat disimpulkan dari aturan yang tersedia.")
        print("Tindakan  : periksa gejala lain atau bawa ke teknisi. (Sistem tidak menebak.)")
        return memori, asal
    print("Diagnosis :", ", ".join(diagnosis))
    print("Saran     :", ", ".join(s.replace("saran_", "").replace("_", " ") for s in saran))
    print("Penjelasan:")
    for d in diagnosis: jelaskan(d, asal, tingkat=1)
    return memori, asal

_ = konsultasi(["layar_gelap", "kipas_berputar", "monitor_eksternal_normal"])
print("\n" + "-" * 70 + "\n")
_ = konsultasi(["lambat", "sering_hang"])

Pada kasus kedua, tidak ada aturan yang cocok. Sistem yang baik **mengakui batas pengetahuannya** dan tidak memaksakan diagnosis. Perilaku ini harus dirancang dengan sengaja.

### Versi interaktif (opsional)
Hapus tanda `#` pada baris terakhir untuk mencoba konsultasi tanya-jawab. Sistem menanyakan gejala satu per satu.

In [ ]:
SEMUA_GEJALA = sorted({p for a in ATURAN for p in a["jika"]} - {a["maka"] for a in ATURAN})

def konsultasi_interaktif():
    gejala = []
    for g in SEMUA_GEJALA:
        if input(f"Apakah {g.replace('_', ' ')}? (y/t) ").strip().lower().startswith("y"):
            gejala.append(g)
    print(); konsultasi(gejala)

print("Gejala yang dapat ditanyakan:", SEMUA_GEJALA)
# konsultasi_interaktif()

---
## Latihan (log implementasi menuju A6)

### Latihan 1 — Telusur manual
Dengan gejala `["mati_mendadak", "bodi_panas", "kipas_berisik"]`:
1. tuliskan di lembar tugas isi memori kerja pada setiap putaran forward chaining (strategi: urutan);
2. gambarkan pohon pembuktian backward chaining untuk hipotesis `saran_bersihkan_pendingin`;
3. cocokkan dengan keluaran kode pada sel di bawah.

In [ ]:
# Jawaban Latihan 1

### Latihan 2 — Mesin inferensi untuk domain Anda
Gunakan domain kelompok dari pertemuan 9.
1. Tulis **minimal 12 aturan** dengan kedalaman rantai minimal 3 (ada kesimpulan antara yang menjadi premis aturan lain). Setiap aturan wajib memiliki `alasan` dan **sumber pengetahuan** (buku, pakar, pedoman resmi).
2. Siapkan **5 kasus uji**: gejala masukan dan kesimpulan yang diharapkan. Minimal satu kasus harus berakhir dengan "belum dapat disimpulkan".
3. Jalankan forward chaining dan backward chaining pada kelima kasus, lalu tampilkan jejak penjelasannya.

In [ ]:
# Jawaban Latihan 2: aturan domain Anda

In [ ]:
# Jawaban Latihan 2: kasus uji

### Latihan 3 — Pengujian otomatis
Lengkapi fungsi `uji(kasus)` yang menjalankan semua kasus uji dan mencetak tabel: nama kasus, kesimpulan yang diharapkan, kesimpulan sistem, dan status LULUS/GAGAL. Tabel inilah cikal bakal *test suite* mini-project Anda.

In [ ]:
def uji(kasus):
    """kasus: list of dict(nama=..., gejala=[...], harapan={...})"""
    # TODO: lengkapi
    pass

### Refleksi
1. Dua aturan pada basis pengetahuan Anda saling bertentangan. Bagaimana cara mendeteksi dan menyelesaikannya?
2. Apa kelemahan utama sistem berbasis aturan dibanding sistem yang belajar dari data? Dan apa keunggulannya dari sisi transparansi?
3. Bila sistem Anda memberi saran yang keliru dan merugikan pengguna, siapa yang bertanggung jawab: penulis aturan, pemrogram, atau pengguna? Jelaskan.

*Jawaban:*

---
**Pengumpulan:** `P10_NIM_Nama.ipynb` diunggah ke LMS sebagai log implementasi.